# Phase 4 - Leakage-Safe Feature Engineering

Prediction scenario: **At the moment a Purchase Order has been created, predict whether the case will eventually breach its SLA.**

This notebook builds one ML observation per case using only information available at or before the first `Purchase Order` timestamp. It does not modify `data/raw/` and does not train any model.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data" / "raw").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.build_features import (
    BUSINESS_FEATURES,
    PROCESS_FEATURES,
    TARGET,
    analyze_features,
    build_feature_dataset,
    load_raw_data,
    write_report,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "p2p_ml_dataset.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

## 1. Load Raw Data

The raw event log and case summary are read into memory. No raw file is modified.

In [ ]:
events, cases = load_raw_data()

pd.DataFrame([
    {"dataset": "p2p_event_log.csv", "rows": len(events), "columns": events.drop(columns=["timestamp_dt"]).shape[1]},
    {"dataset": "p2p_case_summary.csv", "rows": len(cases), "columns": cases.drop(columns=["start_time_dt"]).shape[1]},
])

## 2. Prediction Point

For every case, the prediction point is the first occurrence of `Purchase Order`. The features are computed from the prefix of events whose timestamp is less than or equal to that first Purchase Order timestamp.

In [ ]:
purchase_order_counts = (
    events[events["activity"] == "Purchase Order"]
    .groupby("case_id")
    .size()
    .rename("purchase_order_events")
    .reset_index()
)

pd.DataFrame({
    "cases_in_event_log": [events["case_id"].nunique()],
    "cases_with_purchase_order": [purchase_order_counts["case_id"].nunique()],
    "cases_with_multiple_purchase_orders": [(purchase_order_counts["purchase_order_events"] > 1).sum()],
})

## 3. Build Leakage-Safe Dataset

The builder creates one observation per case and includes only:

- Business attributes known on the Purchase Order event.
- Process-prefix features available at or before the Purchase Order timestamp.
- The eventual `sla_breached` label from the case summary.

Forbidden future/outcome fields such as `end_time`, `duration_hours`, and `duration_days` are excluded.

In [ ]:
ml_dataset, validation = build_feature_dataset(events, cases)
ml_dataset.head()

## 4. Validate Leakage Safety

These checks prove that the dataset has one row per case and that all feature-producing timestamps are at or before the prediction timestamp.

In [ ]:
validation

In [ ]:
assert validation["passed"].all()
assert len(ml_dataset) == ml_dataset["case_id"].nunique()
assert {"end_time", "duration_hours", "duration_days"}.isdisjoint(ml_dataset.columns)
assert len(ml_dataset) == cases["case_id"].nunique()

## 5. Feature Set

The final dataset keeps identifiers and audit columns (`case_id`, `prediction_timestamp`, `purchase_order_event_id`) but these should not be used as model features. The feature columns below are the candidate predictors.

In [ ]:
feature_inventory = pd.DataFrame({
    "feature": BUSINESS_FEATURES + PROCESS_FEATURES + [TARGET],
    "role": ["business"] * len(BUSINESS_FEATURES) + ["process"] * len(PROCESS_FEATURES) + ["target"],
})
feature_inventory

## 6. Save Processed Dataset

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ml_dataset.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

## 7. Target Class Balance

In [ ]:
analysis = analyze_features(ml_dataset)
analysis["class_balance"]

## 8. Numeric Feature Distributions

In [ ]:
analysis["numeric_distribution"]

## 9. Numeric Correlation With Target

These correlations are descriptive only. They should not be interpreted as causal relationships.

In [ ]:
analysis["numeric_target_correlation"]

## 10. Categorical Distributions

In [ ]:
analysis["categorical_distribution"].head(30)

## 11. SLA Breach Rate by Priority and Category

In [ ]:
analysis["breach_by_priority"]

In [ ]:
analysis["breach_by_category"]

## 12. Vendor Analysis and High Cardinality

`vendor_id` is useful but should be handled carefully in modeling because it is a higher-cardinality categorical feature. Avoid naive target encoding unless it is computed out-of-fold or from prior historical data only.

In [ ]:
analysis["vendor_cardinality"]

In [ ]:
analysis["breach_by_vendor"].head(15)

## 13. Write Feature Engineering Report

The report summarizes the prediction point, target, feature definitions, leakage-prevention checks, class balance, and modeling considerations.

In [ ]:
write_report(ml_dataset, validation, analysis)
PROJECT_ROOT / "reports" / "feature_engineering_report.md"

## 14. Modeling Notes for Later

Do not train a model in this phase. For the next phase, use leakage-safe preprocessing: split data before target/frequency encoding, keep audit columns out of the model matrix, and ensure any vendor historical aggregates are computed using only training-fold or prior-case information.